In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import RF_Track as rft
from scipy.optimize import minimize, curve_fit
from partrec_gaussian_optimiser_utils import partrec_gaussian_optimiser_utils
from topasToDose import getDosemap
from uniformity_fit import *
from partrec_foil_plotting import partrec_foil_plotting
from RF_track_utils import *
from flatness import *

In [ ]:
def get_s2_params(max_radius, max_thickness, N_slices, convolution_factor):
    s2_sigma = max_radius / 2
    step = max_radius / N_slices

    # Radial positions
    x = np.arange(-(max_radius + step), 0, step=step)

    # Thickness profile
    y = norm.pdf(x, 0, s2_sigma * convolution_factor)
    y = y - np.min(y)
    y *= max_thickness / np.max(y)
    # plt.scatter(x, y)
    slice_widths = np.diff(y) #thicknesses of each slice in L
    slice_radii = np.abs(x[1:])   # Corresponding radii (height)

    return slice_radii, slice_widths
            

In [ ]:

s2_r = [6.5, 4.9, 3.3, 1.6]
s2_l = [1.4,2.9,4.4,4.3]

s2_r = [3.9  , 2.9, 2 , 1]
s2_l = [0.60, 1.24, 1.85, 1.81]

get_s2_params(3.9,5.5, 4, 1)

In [ ]:

mass = rft.electronmass    # particle mass in MeV/c^2
population = 10 * rft.nC               # number of particles per bunch                         # particle charge in e units
P_ref = 200
N_particles = int(3000000)
charge = -1


# x, y, xp, yp = (rft.qrandn(N_particles, 4) * [1, 1, 0.01, 0.01]).T


# P = np.ones(N_particles) * P_ref
# T = np.zeros(N_particles)
# matrix = np.column_stack((x, xp, y, yp, T, P)) #transpose to match Bunch6d format
# B0 = rft.Bunch6d(mass, N_particles, charge, matrix)

Twiss = rft.Bunch6d_twiss()

if P_ref == 150:
    Twiss.beta_x = 43.6        # m
    Twiss.beta_y = 43.6      # m
    Twiss.emitt_x = 17.3/1000 * (P_ref / mass)    # mm.mrad normalised emittance
    Twiss.emitt_y = 12.3/1000 * (P_ref / mass)     # mm.mrad

elif P_ref == 200:
    Twiss.beta_x = 60        # m
    Twiss.beta_y = 80      # m
    Twiss.emitt_x = 6.75    # mm.mrad normalised emittance
    Twiss.emitt_y = 6.75     # mm.mrad    

elif P_ref == 250:
    Twiss.beta_x = 72.6        # m
    Twiss.beta_y = 72.6      # m 
    Twiss.emitt_x = 6.75    # mm.mrad normalised emittance
    Twiss.emitt_y = 6.75     # mm.mrad


# Twiss.sigma_t = 10 * RF_Track.ps       # mm/c   or 37 * RF_Track.ps
# Twiss.sigma_pt = 10     # permille
Twiss.mean_xp = 0.0
Twiss.mean_yp = 0.0

B0 = rft.Bunch6d_QR(mass, population, charge, P_ref, Twiss, N_particles)      








In [ ]:
best_seen = [np.inf, None]



def loss (params, B0, target_size=6):  #16 target good for 10mm, 6 good for 5mm
    # s1_l, s2_width, s2_depth = params
    s2_width, s2_depth = params
    '''params dimensions in mm'''
    lattice = rft.Lattice()
    window = rft.Absorber(250e-6,'beryllium')
    window.disable_energy_straggling()
    window.set_shape ('circular', 0.5,0.5 )
    lattice.append(window)
    lattice.append(rft.Drift(0.5))  #drift to s1
    
    
#     S1 = rft.Absorber(s1_l/1000,8.897, 13,26.982,2.7, 166)
# # S1 = rft.Absorber(0.0001,'air')
#     S1.disable_energy_straggling()
#     S1.set_shape ('circular', 1,1  )
    
    # lattice.append(S1)
    

    lattice.append(rft.Drift(0.5))  #drift to S2
    # lattice.append(rft.Drift(0.532))  #drift to S2

    s2_r,s2_l = get_s2_params(s2_width, s2_depth, 4, 1)
  
    for i in range(len(s2_l)):
        Slice = rft.Absorber(s2_l[i]/1000,31.9, 37, 288.31,1.32,-1)
        Slice.disable_energy_straggling()
        Slice.set_shape ('circular',  abs(s2_r[i])/1000,abs(s2_r[i])/1000 )
        # Slice.set_shape ('circular', 2,2 )
        lattice.append(Slice)

    lattice.append(rft.Drift(0.5))  #drift to phsp
    # lattice.append(rft.Drift(2.204))

    B1 = lattice.track(B0)
    M = B1.get_phase_space('%x %xp %y %yp %E %z')
   
    x = M[:,0]
    y = M[:,2]
    count = np.sum(x**2 + y**2 < 10**2)
    transmission_10mm = count / len(x)
    

    # loss = flatness(hist_x) + flatness(hist_y)
    # loss = merit_beam_Uniform(B1, 5, 20, transmission=0.998)
    masked_x, masked_y = mask2d(M[:,0],M[:,2])
    


    # loss = abs(params_x[1]/params_x[2] -1.1)  + abs(params_y[1]/params_y[2] -1.1) #x0/sigma  
    # loss = loss_2gauss(hist_x, bin_centers_x, hist_y, bin_centers_y)
   
    # print(f'masked_x_range:{max(masked_x)},{min(masked_x)}, masked_y_range:{max(masked_y)},{min(masked_y)}'),   
    # loss = nearest_neighbor_test(masked_x,masked_y)[2] # + (1-transmission_10mm)*0.1
   
    # print(f'masked_x_range:{max(masked_x)},{min(masked_x)}, masked_y_range:{max(masked_y)},{min(masked_y)}'),   
    loss = nearest_neighbor_test(masked_x,masked_y)[2] + 0.5 * abs(masked_x.max()/target_size-1) #+ abs(len(M))/int(B0.get_population()) - 1))
    print('nn contribution:', nearest_neighbor_test(masked_x,masked_y)[2])
    print('size contribution:', abs(masked_x.max()/target_size-1))
    print('x range:',max(masked_x),min(masked_x),'y range:', max(masked_y),min(masked_y))
    # slice_width = 1
    if loss < best_seen[0]:
        best_seen[0] = loss
        best_seen[1] = params.copy()

    # print('loss:', loss, 's1_l:', s1_l, 's2_2r:', s2_width, 's2_l:', s2_depth)
    print('loss:', loss, 's2_2r:', s2_width, 's2_l:', s2_depth)
    return loss



best_seen = [np.inf, None]



def loss (params, B0, target_size=10):  #16 target good for 10mm, 6 good for 5mm
    s1_l, s2_width, s2_depth = params
    '''params dimensions in mm'''
    lattice = rft.Lattice()
    window = rft.Absorber(250e-6,'beryllium')
    window.disable_energy_straggling()
    window.set_shape ('circular', 0.5,0.5 )
    lattice.append(window)
    lattice.append(rft.Drift(0.5))  #drift to s1
    
    S1 = rft.Absorber(s1_l/1000,8.897, 13,26.982,2.7, 166)
# S1 = rft.Absorber(0.0001,'air')
    S1.disable_energy_straggling()
    S1.set_shape ('circular', 1,1  )
    
    lattice.append(S1)
    

    lattice.append(rft.Drift(0.5))  #drift to S2
    # lattice.append(rft.Drift(0.532))  #drift to S2

    s2_r,s2_l = get_s2_params(s2_width, s2_depth, 4, 1)
  
    for i in range(len(s2_l)):
        Slice = rft.Absorber(s2_l[i]/1000,31.9, 37, 288.31,1.32,-1)
        Slice.disable_energy_straggling()
        Slice.set_shape ('circular',  abs(s2_r[i])/1000,abs(s2_r[i])/1000 )
        # Slice.set_shape ('circular', 2,2 )
        lattice.append(Slice)

    lattice.append(rft.Drift(0.5))  #drift to phsp
    # lattice.append(rft.Drift(2.204))

    B1 = lattice.track(B0)
    M = B1.get_phase_space('%x %xp %y %yp %E %z')
   
    x = M[:,0]
    y = M[:,2]
    count = np.sum(x**2 + y**2 < target_size**2)
    transmission_10mm = count / len(x)
    

    # loss = flatness(hist_x) + flatness(hist_y)
    # loss = merit_beam_Uniform(B1, 5, 20, transmission=0.998)
    masked_x, masked_y = mask2d(M[:,0],M[:,2])
    r = np.sqrt(x**2 + y**2)
    mask = r <= target_size*1.3
    


    slice_width = 1
    n_bins,fov  = 15,250
    phsp_xslice = M[(M[:,2] < slice_width)]
    phsp_xslice = phsp_xslice[(phsp_xslice[:,2] > -slice_width)]
    
    phsp_yslice = M[(M[:,0] < slice_width)]
    phsp_yslice = phsp_yslice[(phsp_yslice[:,0] > -slice_width)]
    
    
    hist_x, bin_edges_x = np.histogram(phsp_xslice[:,0], bins=n_bins, range=[-fov, fov])
    bin_centers_x = (bin_edges_x[:-1] + bin_edges_x[1:]) / 2
    hist_y, bin_edges_y = np.histogram(phsp_yslice[:,2], bins=n_bins, range=[-fov, fov])
    bin_centers_y = (bin_edges_y[:-1] + bin_edges_y[1:]) / 2

    p0=[np.max(hist_x),  np.mean(phsp_xslice[:,0]), np.std(phsp_xslice[:,0]), 4]
    params_x, _ = curve_fit(supergaussian1D, bin_centers_x, hist_x, p0=p0)
    params_y, _ = curve_fit(supergaussian1D, bin_centers_y, hist_y, p0=p0) 
    
    r90_x = r90(params_x[2], params_x[3])
    r90_y = r90(params_y[2], params_y[3])
    # loss = abs(params_x[1]/params_x[2] -1.1)  + abs(params_y[1]/params_y[2] -1.1) #x0/sigma  
    # loss = loss_2gauss(hist_x, bin_centers_x, hist_y, bin_centers_y)
   
    # print(f'masked_x_range:{max(masked_x)},{min(masked_x)}, masked_y_range:{max(masked_y)},{min(masked_y)}'),   
    # loss = nearest_neighbor_test(masked_x,masked_y)[2] + abs(masked_x.max()/target_size-1)
    # loss = nearest_neighbor_test(x[mask],y[mask])[2] + 2*abs(1-r90_x/target_size) + abs(1-r90_y/target_size) + (1-transmission_10mm)*0.1
   
    # print(f'masked_x_range:{max(masked_x)},{min(masked_x)}, masked_y_range:{max(masked_y)},{min(masked_y)}'),   
    loss = nearest_neighbor_test(masked_x,masked_y)[2] + 0.5 * abs(masked_x.max()/target_size-1) #+ abs(len(M))/int(B0.get_population()) - 1))
    print('nn contribution:', nearest_neighbor_test(masked_x,masked_y)[2])
    print('size contribution:', abs(masked_x.max()/target_size-1))
    print('x range:',max(masked_x),min(masked_x),'y range:', max(masked_y),min(masked_y))
    # slice_width = 1
    if loss < best_seen[0]:
        best_seen[0] = loss
        best_seen[1] = params.copy()

    print('loss:', loss, 's1_l:', s1_l, 's2_2r:', s2_width, 's2_l:', s2_depth)
    return loss

In [ ]:
# B0_opt = rft.Bunch6d(mass, 10000, charge, matrix)
B0_opt = rft.Bunch6d_QR(mass, population, charge, P_ref, Twiss, 10000) 
rng = np.random.default_rng(12345)
x0 = rng.uniform(
    low=[0.01, 0.01],
    high=[ 10, 10]
)

# x0 = [ 3.6, 4.7]

opt_loss = np.inf
best_seen = [np.inf, None]

for i in range(5):
    res = minimize(loss,
                      x0=x0, args=(B0_opt,),
                      bounds=[ (0.01,20),(0.01,20)], #absorber thickness 0 breaks rf track
                      method='Powell',
                      options={'disp': True
                               , 'xtol':0.01}
                    #   tol=1e-2
                      )
    if res.fun < opt_loss:
        opt_loss = res.fun
        opt_params = res.x

print(f'opt_loss={opt_loss:.3f}, params, ={opt_params}')
print('best seen:', best_seen)

In [ ]:
from scipy.optimize import differential_evolution

res = differential_evolution(
    loss,
    bounds=[(0.01,5),(0.01,20),(0.01,20)],
    args=(B0_opt,)
)

print(f'loss={res.fun:.3f}, params, ={res.x}')


In [ ]:
best_seen, loss(best_seen[1], B0_opt)

In [ ]:
loss(opt_params, B0_opt)

In [ ]:

# s1_l, s2_width, s2_depth = res.x
# s1_l, s2_width, s2_depth =   best_seen[1]
# s2_width, s2_depth = best_seen[1]
s1_l, s2_width, s2_depth = 0.3 ,3.9,5.3

# for s2_width in [3.3,3.5, 3.7, 4.0]:
#     for s2_depth in [4.6,4.8, 5.0]:
lattice = rft.Lattice()
window = rft.Absorber(250e-6,'beryllium')
window.disable_energy_straggling()
window.set_shape ('circular', 0.5,0.5 )
lattice.append(window)  

lattice.append(rft.Drift(0.5))  #drift to s1

S1 = rft.Absorber(s1_l/1000,8.897, 13,26.982,2.7, 166)
# S1 = rft.Absorber(0.0001,'air')
S1.disable_energy_straggling()
S1.set_shape ('circular', 1,1 )
lattice.append(S1)

lattice.append(rft.Drift(0.5))  #drift to S2
# lattice.append(rft.Drift(0.532))  #drift to S2
s2_r,s2_l = get_s2_params(s2_width, s2_depth, 4, 1)
# s2_r = [3.7, 2.8, 1.9, 0.9]
# s2_l = [0.5, 1.1 , 1.6, 1.6]

for i in range(len(s2_l)):
    Slice = rft.Absorber(s2_l[i]/1000,31.9, 37, 288.31,1.32,-1)
    Slice.disable_energy_straggling()
    Slice.set_shape ('circular',  abs(s2_r[i])/1000,abs(s2_r[i])/1000 )
    lattice.append(Slice)

lattice.append(rft.Drift(0.5))  #drift to phsp


B1 = lattice.track(B0)
T = lattice.get_transport_table(
'%S %beta_x %beta_y %alpha_x %alpha_y %sigma_x %sigma_y %sigma_px %sigma_py')
M = B1.get_phase_space('%x %xp %y %yp %E %z')
x = M[:,0]
y = M[:,2]
count = np.sum(x**2 + y**2 < 10**2)
transmission_10mm = count / len(x)
count = np.sum(x**2 + y**2 < 5**2)
transmission_5mm = count / len(x)
plot_phsp(T,M,120,10,slice_width=0.1)
print('tranmission 10mm:', transmission_10mm, 'transmission 5mm:', transmission_5mm)

In [ ]:
loss(opt_params, B0, target_size=7.5)

checking results

In [ ]:
# B0.get_population()
len(M)

checking with TOPAS

In [ ]:
#twiss to 6d params
if P_ref == 150:
    Twiss.beta_x = 43.6        # m
    Twiss.beta_y = 43.6      # m
    geo_emitt_x = 22.9/1000
    geo_emitt_y = 22.9/1000

elif P_ref == 200:
    Twiss.beta_x = 60        # m
    Twiss.beta_y = 80      # m
    geo_emitt_x = 17.3/1000
    geo_emitt_y = 12.3/1000

elif P_ref == 250:
    Twiss.beta_x = 72.6        # m
    Twiss.beta_y = 72.6      # m 
    geo_emitt_x = 13.8/1000
    geo_emitt_y = 13.8/1000

Twiss.alpha_x = 0
Twiss.alpha_y = 0


beta_gamma = P_ref / mass

Twiss.emitt_x = geo_emitt_x * beta_gamma    # mm.mrad normalised emittance
Twiss.emitt_y = geo_emitt_y* beta_gamma     # mm.mrad

sig_x = np.sqrt(Twiss.beta_x * geo_emitt_x)
sig_y = np.sqrt(Twiss.beta_y * geo_emitt_y)
sig_xp = max(0.01, Twiss.alpha_x * np.sqrt(geo_emitt_x / Twiss.beta_x))
sig_yp = max(0.01, Twiss.alpha_y * np.sqrt(geo_emitt_y / Twiss.beta_y))
print('sig_x:', sig_x, 'sig_y:', sig_y, 'sig_xp:', sig_xp, 'sig_yp:', sig_yp)
beta_gamma

# sig_x = 0.15726410906497387
# sig_y = 0.16368567438844486 
# sig_xp = 2.133359740414439 
# sig_yp = 2.2285038218591913

In [ ]:
output_filename='CLARA_dualscattering_5mm_25mmtank'
dose_depth=5
dir = '/Users/sabrinawang/Desktop/DPhil_Project/'
profile = "dose"
N_particles = 3000000

# s1_l, s2_width, s2_depth = 2.27403378, 3.80651473, 4.52916474 '''works great'''

s1_l, s2_width, s2_depth = 0.3, 3.9, 5.3

setup = partrec_gaussian_optimiser_utils(world_material="Air")

setup.generate_phsp_beam(sig_x,sig_y,sig_xp,sig_yp,P_ref,0,N_particles)

setup.add_cylinder("window",0.25,0,500,"G4_Be",1)

# s1_pos = 510
s1_pos = 501
setup.add_flat_scatterer(s1_l,"Aluminum",s1_pos)

# setup.add_gaussian_scatterer(s2_depth,s2_width,1,4,"PEEK",s1_pos + 500)

s2_r = [3.9  , 2.9, 2 , 1]
s2_l = [0.60, 1.24, 1.85, 1.81]

s2_r = s2_r[::-1]
s2_l = s2_l[::-1]
# constructing s2 from slices

# s2_pos = s1_pos + 490 - sum(s2_l) #position of base of s2 (largest slice)
s2_pos = s1_pos + 500

for i in range(len(s2_l)):
    sname = 'S2_slice_' + str(i)
    slice_position = s2_pos + sum(s2_l[:i-1])   # sum of previous lengths
    setup.add_cylinder(sname, s2_l[i-1], 0, s2_r[i-1], 'PEEK', slice_position)
    
# setup.add_cylinder('kapton',0.25, 0,33,"kapton",s2_pos + sum(s2_l))

# setup.add_box('collimator',74,50,152.4,'G4_Pb',slice_position+80)
# setup.add_cylinder('collimator_hole',74, 0,5,"Air",slice_position+80)

# setup.add_cylinder('tank_edge',1, 0,100,"PMMA",1513) #tank at 1512 + 1 for berylium window

# if profile == "dose":
#     setup.add_tank_bins(1514, dose_depth, 100,100,1,output_filename,width=30)
# elif profile == "intensity":
#     setup.add_patient(1514)

if profile == "dose":
    setup.add_tank_bins(s1_pos + 1000, dose_depth, 100,100,1,output_filename,width=20)
elif profile == "intensity":
    setup.add_patient(s1_pos + 1000)


setup.run_topas(view_setup=False)   




In [ ]:


if profile == "dose":
#             # initialise plotting class
    # doseMap = getDosemap("DoseAtTank"+str(dose_depth)+ "_"+ output_filename+".csv",N_particles, dose_depth, output_filename, plot = True) 
    # fitDoseMap(N_particles, dose_depth,output_filename)
    x,y, doseMap = getDosemap("DoseAtTank"+str(dose_depth)+ "_"+ output_filename+".csv",N_particles, dose_depth, output_filename, plot = True)
    plot_dose(doseMap, x, y)

elif profile == "intensity":
    plotter = partrec_foil_plotting('patient_beam.phsp' ) #filename defined inside partrec_gaussian_optimiser_utils
    # plot transverse distributions and energy spectrum at patient
    plotter.show_transverse_beam(output_filename, s1_l, s2_depth, s2_width,particle= 'e',fov= 18, col=5)
    plotter.show_transverse_beam(output_filename, s1_l, s2_depth, s2_width,particle= 'y',fov=18, col=5)